<a href="https://colab.research.google.com/github/arnabd64/Google-Colab-Notebooks/blob/main/Movie_Genre_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Movie Genre Prediction

In [3]:
! pip install --progress-bar=off \
    transformers[torch] \
    accelerate \
    evaluate \
    tensorboard \
    wandb \
> install.log

## Login to Huggingface and Weights & Biases

In [ ]:
from google.colab import userdata
import huggingface_hub as hub
import wandb

hub.login(userdata.get("HF"))
wandb.login(key=userdata.get("WANDB"))

Token will not been saved to git credential helper. Pass `add_to_git_credential=True` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /root/.cache/huggingface/token
Login successful


wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

## Set Dataset & Base Model

In [ ]:
DATASET = "datadrivenscience/movie-genre-prediction"
BASE_MODEL = "gpt2"
TUNED_MODEL = "GPT2-genre-detection"

## Load Dataset

In [ ]:
from datasets import load_dataset, concatenate_datasets

genre = concatenate_datasets([
    load_dataset(DATASET, split="train"),
    load_dataset(DATASET, split="test")
])

genre = genre.train_test_split(train_size=0.9, shuffle=True)
genre = genre.select_columns(['synopsis','genre'])
genre = genre.rename_columns({"synopsis":"text", "genre":"label"})

Generating train split:   0%|          | 0/54000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/36000 [00:00<?, ? examples/s]

## Evaluation Function

In [ ]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")
cola = evaluate.load("glue", "cola")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)

    accuracy_metric = accuracy.compute(predictions=predictions, references=labels)
    cola_metric = cola.compute(predictions=predictions, references=labels)

    return {"accuracy": accuracy_metric["accuracy"], "matthews_correlation": cola_metric["matthews_correlation"]}

## Load Model & Tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

## Set Labels

In [ ]:
LABELS = list(set(genre['train']['label']))
LABEL2ID = {label:id for id, label in enumerate(LABELS)}
ID2LABEL = {id:label for id, label in enumerate(LABELS)}

## Preprocessing functions

In [ ]:
def tokenize(batch):
    return tokenizer(batch['text'], padding="max_length", max_length=256)

def convert_labels_to_ids(batch):
    batch['label'] = [LABEL2ID[label] for label in batch['label']]
    return batch


genre = genre.map(convert_labels_to_ids, batched=True)
genre = genre.map(tokenize, batched=True)

Map:   0%|          | 0/81000 [00:00<?, ? examples/s]

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/81000 [00:00<?, ? examples/s]

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

## Setup Training

In [ ]:
from transformers import Trainer, TrainingArguments

def get_model(model):
    model = AutoModelForSequenceClassification.from_pretrained(model, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID)
    model.config.pad_token_id = model.config.eos_token_id
    return model


args = TrainingArguments(
    output_dir = TUNED_MODEL,
    save_safetensors = True,
    logging_strategy="steps",
    logging_steps=100,
    evaluation_strategy = "epoch",
    eval_steps = 1,
    save_strategy = "epoch",
    save_steps = 1,
    num_train_epochs = 2,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 32,
    learning_rate = 2e-5,
    warmup_ratio = 0.05,
    lr_scheduler_type = 'cosine',
    report_to = ['tensorboard', 'wandb'],
    load_best_model_at_end = True,
    metric_for_best_model = "matthews_correlation",
    greater_is_better = True,
    seed = generate_random_seed(8)
)

trainer = Trainer(
    model = get_model(BASE_MODEL),
    args = args,
    tokenizer = tokenizer,
    train_dataset = genre['train'],
    eval_dataset = genre['test'],
    compute_metrics = compute_metrics
)

# output = trainer.train()

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Run training with Weights & Biases for Realtime Monitoring

In [ ]:
run = wandb.init(project="Movie-Genre-Prediction", name=TUNED_MODEL)
try:
    trainer.train()
    run.finish()

except Exception as e:
    print(e)
    run.finish()

wandb: Currently logged in as: 2wb2ndur. Use `wandb login --relogin` to force relogin


You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Accuracy,Matthews Correlation
1,1.576500,1.556155,0.458889,0.089942
2,1.505800,1.526665,0.459333,0.101033


eval/accuracy,▁█
eval/loss,█▁
eval/matthews_correlation,▁█
eval/runtime,█▁
eval/samples_per_second,▁█
eval/steps_per_second,▁█
train/epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
train/global_step,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
train/learning_rate,▂▅███████▇▇▇▇▇▆▆▆▆▅▅▅▄▄▄▄▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁
train/loss,█▅▃▃▂▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂▁▁▁▂▂▂▁▁▂▁▂▁▁▁▁▂▂
train/total_flos,▁


## Push trained model to Huggingface

In [ ]:
trainer.push_to_hub(language="en", dataset=DATASET, tags=['pytorch', trainer.model.__class__.__name__])

training_args.bin:   0%|          | 0.00/4.73k [00:00<?, ?B/s]

events.out.tfevents.1703314564.0664e86b35ce.711.0:   0%|          | 0.00/38.0k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

Upload 3 LFS files:   0%|          | 0/3 [00:00<?, ?it/s]

'https://huggingface.co/arnabdhar/GPT2-genre-detection/tree/main/'